# 02 — Modeling

Goal: run experiments and evaluate with proper CV. Log each run to `memory/experiments.md`.

Keep logic that you will reuse in `src/` (features.py, models.py, utils.py).
Notebooks are for orchestration and inspection, not for the canonical implementation.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.utils import load_train, load_test, load_sample_submission, save_submission, PATHS
from src import features as F
from src import models as M

In [3]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.model_selection import StratifiedKFold



In [5]:
# ── paths & data ─────────────────────────────────────────────────────────────
DATA_DIR = Path.cwd().parent / "data"
df_train = pd.read_csv(DATA_DIR / "train.csv")

NUM_COLS = [
    "Soil_Moisture", "Temperature_C", "Rainfall_mm", "Wind_Speed_kmh",
    "Humidity", "Sunlight_Hours", "Crop_Water_Need_mm",
    "Field_Area_hectares", "Elevation_m", "Soil_pH", "Fertilizer_Used_kg",
]
CAT_COLS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]
TARGET = "Irrigation_Need"
CLASSES = ["Low", "Medium", "High"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}



# ── encode target ─────────────────────────────────────────────────────────────
X = df_train[NUM_COLS + CAT_COLS].copy()
for c in CAT_COLS:
    X[c] = X[c].astype("category")
y = df_train[TARGET].map(CLASS_TO_IDX).to_numpy()

# ── class weights (inverse frequency, mean=1) ────────────────────────────────
counts = np.bincount(y, minlength=3)
class_w = len(y) / (3 * counts)
sample_w = class_w[y]

# ── 5-fold stratified CV ──────────────────────────────────────────────────────
lgbm_params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "lambda_l2": 1.0,
    "verbose": -1,
    "seed": 42,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros((len(X), 3), dtype=np.float32)

print("Training 5-fold LightGBM (Experiment #001 config)…")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    w_tr, w_va = sample_w[tr_idx], sample_w[va_idx]

    dtr = lgb.Dataset(X_tr, label=y_tr, weight=w_tr, categorical_feature=CAT_COLS)
    dva = lgb.Dataset(X_va, label=y_va, weight=w_va, categorical_feature=CAT_COLS, reference=dtr)

    model = lgb.train(
        lgbm_params, dtr,
        num_boost_round=2000,
        valid_sets=[dva],
        valid_names=["val"],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    preds = model.predict(X_va, num_iteration=model.best_iteration)
    oof[va_idx] = preds
    fold_score = balanced_accuracy_score(y_va, preds.argmax(axis=1))
    print(f"  fold {fold+1}/5: BalAcc={fold_score:.5f}  best_iter={model.best_iteration}")

oof_score = balanced_accuracy_score(y, oof.argmax(axis=1))
per_class = recall_score(y, oof.argmax(axis=1), average=None)
print(f"\nOOF BalAcc (argmax): {oof_score:.5f}")
print(f"Per-class recall  — Low: {per_class[0]:.4f}  Medium: {per_class[1]:.4f}  High: {per_class[2]:.4f}")

# ── post-hoc per-class log-offset tuning ────────────────────────────────────
eps   = 1e-12
logp  = np.log(oof + eps)
offsets = np.zeros(3)
best_v  = balanced_accuracy_score(y, logp.argmax(axis=1))
grid    = np.linspace(-1.5, 1.5, 31)

for _ in range(3):
    improved = False
    for k in range(3):
        base = offsets.copy()
        for g in grid:
            base[k] = g
            v = balanced_accuracy_score(y, (logp + base).argmax(axis=1))
            if v > best_v:
                best_v, offsets, improved = v, base.copy(), True
    if not improved:
        break

print(f"Tuned BalAcc      : {best_v:.5f}  (offsets {offsets.round(1).tolist()})")


KeyError: "['Crop_Water_Need_mm', 'Field_Area_hectares', 'Elevation_m', 'Fertilizer_Used_kg'] not in index"